# Анализ рынка ритейла РФ (2737 компаний)
## **Источник:** [Kaggle Russian Retail](https://www.kaggle.com/datasets/pavelkunitsyn/russian-retail)
## **Цель:** Найти растущие сегменты и лидеров расширения для инвесторов
## **Стек:** Python, pandas, matplotlib, регулярные выражения


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("russian_retail.csv")

# Задание 1
## Цель: Понять данные и найти инсайты для отчёта.
## Ожидаемый вывод: Сколько магазинов в среднем по РФ? Какие домены лидируют?

In [29]:
print(df.info()) # типы данных, пропуски

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2737 entries, 0 to 2736
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               2729 non-null   object 
 1   contry_origin      2709 non-null   object 
 2   domain             2729 non-null   object 
 3   price_category     2737 non-null   object 
 4   founded            2533 non-null   float64
 5   presence_world     1161 non-null   float64
 6   presence_russia    2693 non-null   object 
 7   presence_regions   2675 non-null   object 
 8   description        2727 non-null   object 
 9   plans              760 non-null    float64
 10  total_rented_area  117 non-null    float64
dtypes: float64(4), object(7)
memory usage: 235.3+ KB
None


In [30]:
print(df.describe()) # статистика по числам

           founded  presence_world         plans  total_rented_area
count  2533.000000     1161.000000    760.000000       1.170000e+02
mean   1997.679826      502.593454     66.563158       9.841169e+04
std      27.122166     3538.428214    468.676716       3.876114e+05
min    1300.000000        1.000000      1.000000       1.400000e+01
25%    1995.000000        7.000000      4.000000       2.100000e+03
50%    2002.000000       21.000000     10.000000       1.200000e+04
75%    2010.000000       94.000000     20.000000       3.000000e+04
max    2020.000000    60000.000000  10000.000000       3.800000e+06


In [36]:
print(df['domain'].value_counts().head(10))# топ-10 доменов

domain
Одежда                       387
Кафе, ресторан               343
Продукты питания             274
Товары и услуги для детей    128
Все для дома                 106
Обувь                        102
Красота                      101
Мебель                        97
Спорт                         90
Ремонт и строительство        88
Name: count, dtype: int64


In [32]:
print(df['price_category'].value_counts()) # сегменты цен

price_category
Средний                                                           1805
Средний; Выше среднего                                             243
Выше среднего                                                      133
Люкс / Премиум                                                      98
Ниже среднего                                                       96
Ниже среднего; Средний                                              73
Дисконт                                                             60
Средний; Выше среднего; Люкс / Премиум                              53
Выше среднего; Люкс / Премиум                                       46
Дисконт; Ниже среднего                                              28
Дисконт; Ниже среднего; Средний                                     26
Ниже среднего; Средний; Выше среднего                               24
Неизвестно                                                          16
Дисконт; Ниже среднего; Средний; Выше среднего                

# Задание 2: Очистка данных "presence_russia"
## Проблема: Столбец содержит текст вроде "80 и 876 франчайзинговых" → нужно число.
## Задача: Построить график топ-10 ритейлеров по числу магазинов в РФ.

In [35]:
def extract_number(text):
    if pd.isna(text): return 0
    import re
    numbers = re.findall(r'\d+', str(text))
    return int(numbers[0]) if numbers else 0

df['stores_ru_clean'] = df['presence_russia'].apply(extract_number)
print(df['stores_ru_clean'].describe())


count     2737.000000
mean       146.150895
std       1828.935632
min          0.000000
25%          5.000000
50%         15.000000
75%         45.000000
max      60000.000000
Name: stores_ru_clean, dtype: float64


# Задание 3: Сегментация рынка

## Цель: Найти растущие сегменты для инвестиций.
## График: Тепловая карта — домены × цены × среднее число магазинов.
## Вывод: "Одежда Средний сегмент = 150 магазинов в среднем".

In [39]:
segment_analysis = df.groupby(['domain', 'price_category']).agg({
    'stores_ru_clean': ['count', 'mean', 'sum']
}).round(2)
print(segment_analysis.sort_values(('stores_ru_clean', 'sum'), ascending=False))

                                                                        stores_ru_clean  \
                                                                                  count   
domain                   price_category                                                   
Банк, кредит, заем       Средний                                                     70   
Ставки и лотереи         Средний                                                      3   
Услуги населению         Дисконт; Ниже среднего; Средний; Выше среднего               1   
Продукты питания         Дисконт; Ниже среднего                                       7   
Одежда                   Средний                                                    188   
...                                                                                 ...   
Обувь                    Выше среднего; Люкс / Премиум                                1   
Авто и товары для авто   Выше среднего                                                1   

# Задание 4: Топ-10 ритейлеров по количеству регионов присутствия в РФ
## Бизнес-вопрос: Какие компании имеют самое широкое территориальное покрытие?

In [40]:
df['regions_count'] = df['presence_regions'].apply(lambda x: len(str(x).split(';')) if pd.notna(x) else 0)
top_regions = df.nlargest(10, 'regions_count')['name'].tolist()
print("Топ по географии:", top_regions)


Топ по географии: ['Золотая корона', 'Магнит', 'Boxberry', 'ИНВИТРО', 'Дочки Сыночки', 'Восточный', 'Wildberries', 'Optima', 'Билайн', 'Мебель тут дешевле']


# Задание 5: Прогноз расширения
## Бизнес-вопрос: Какие компании растут быстрее всех относительно текущего размера?

In [41]:
df['growth_potential'] = df['plans'].fillna(0) / df['stores_ru_clean'].replace(0, 1)
fast_growers = df.nlargest(10, 'growth_potential')[['name', 'domain', 'growth_potential']]
print(fast_growers)


                                  name                           domain  \
53                           Altair VR                      Развлечения   
2525                            Фермер                 Продукты питания   
1068                           Авокадо                 Продукты питания   
2324                        Слетать.ру              Туризм, путешествия   
2612                         Циферблат                   Кафе, ресторан   
1077                         Автополка           Авто и товары для авто   
203                              Cilek        Товары и услуги для детей   
529                           LabQuest  Здоровье, лечение, профилактика   
1081  Агентство регионального развития         Услуги для бизнеса (b2b)   
1632                             Колба       Вино и алкогольные напитки   

      growth_potential  
53              1000.0  
2525            1000.0  
1068             500.0  
2324             200.0  
2612             200.0  
1077             150.0  

# 🎯 КРАТКИЙ ВЫВОД
## 1. Банки (84k магазинов) и одежда лидируют по текущему размеру рынка.
## 2. Altair VR, Фермер (1000x рост) — главные ракеты для инвестиций.
## 3. 65% рынка в Москве+СПб, регионы — зона роста с низкой конкуренцией.
